<a href="https://colab.research.google.com/github/yaswanthvuppala/NLP-RAG/blob/main/Embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import time
import numpy as np
from sentence_transformers import SentenceTransformer


# --------------------------------------------------
# 1. Fixed evaluation dataset
# Keep these documents and questions unchanged
# when comparing models.
# --------------------------------------------------

documents = [
    {
        "id": "D1",
        "text": (
            "HTTP 401 indicates missing, invalid, or expired "
            "authentication credentials. Verify the bearer token."
        ),
    },
    {
        "id": "D2",
        "text": (
            "HTTP 403 indicates that an authenticated caller "
            "does not have permission to access a resource."
        ),
    },
    {
        "id": "D3",
        "text": (
            "An embedding represents text as a numerical vector. "
            "Cosine similarity compares the directions of vectors."
        ),
    },
    {
        "id": "D4",
        "text": (
            "BM25 ranks documents using term frequency, inverse "
            "document frequency, and document-length normalization."
        ),
    },
    {
        "id": "D5",
        "text": (
            "Chunking divides long documents into smaller passages. "
            "Overlap can preserve context across chunk boundaries."
        ),
    },
    {
        "id": "D6",
        "text": (
            "Recall at K measures the fraction of relevant documents "
            "retrieved within the first K ranked results."
        ),
    },
]

queries = [
    {
        "query": "How do I troubleshoot an expired bearer token?",
        "relevant_ids": {"D1"},
    },
    {
        "query": "Why does an authenticated user get forbidden?",
        "relevant_ids": {"D2"},
    },
    {
        "query": "How can I measure the similarity between text vectors?",
        "relevant_ids": {"D3"},
    },
    {
        "query": "How does lexical ranking account for rare terms?",
        "relevant_ids": {"D4"},
    },
    {
        "query": "Why split documents and repeat text near boundaries?",
        "relevant_ids": {"D5"},
    },
]


# --------------------------------------------------
# 2. Model-specific text formatting
# MiniLM uses raw text in this experiment.
# E5 uses its documented query/passage prefixes.
# --------------------------------------------------

MODEL_CONFIGS = {
    "MiniLM": {
        "model_id": "sentence-transformers/all-MiniLM-L6-v2",
        "query_prefix": "",
        "document_prefix": "",
    },
    "E5-small": {
        "model_id": "intfloat/multilingual-e5-small",
        "query_prefix": "query: ",
        "document_prefix": "passage: ",
    },
}


# --------------------------------------------------
# 3. Ranking metrics
# --------------------------------------------------

def recall_at_k(ranked_ids, relevant_ids, k):
    if not relevant_ids:
        return 0.0

    retrieved = set(ranked_ids[:k])
    return len(retrieved & relevant_ids) / len(relevant_ids)


def reciprocal_rank(ranked_ids, relevant_ids):
    for rank, doc_id in enumerate(ranked_ids, start=1):
        if doc_id in relevant_ids:
            return 1.0 / rank

    return 0.0


# --------------------------------------------------
# 4. Benchmark one model
# --------------------------------------------------

def benchmark_model(name, config):
    print(f"\nLoading {name}: {config['model_id']}")
    model = SentenceTransformer(config["model_id"])

    doc_texts = [
        config["document_prefix"] + doc["text"]
        for doc in documents
    ]

    start = time.perf_counter()

    doc_vectors = model.encode(
        doc_texts,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=False,
    )

    indexing_seconds = time.perf_counter() - start

    recalls = []
    reciprocal_ranks = []

    for item in queries:
        query_text = config["query_prefix"] + item["query"]

        start = time.perf_counter()

        query_vector = model.encode(
            query_text,
            normalize_embeddings=True,
            convert_to_numpy=True,
            show_progress_bar=False,
        )

        query_seconds = time.perf_counter() - start

        # Normalized vectors make dot product equivalent
        # to cosine similarity.
        scores = doc_vectors @ query_vector

        ranked_indices = np.argsort(scores)[::-1]
        ranked_ids = [
            documents[int(i)]["id"]
            for i in ranked_indices
        ]

        relevant_ids = item["relevant_ids"]

        recalls.append(
            recall_at_k(ranked_ids, relevant_ids, k=3)
        )
        reciprocal_ranks.append(
            reciprocal_rank(ranked_ids, relevant_ids)
        )

        print(f"\nQuestion: {item['query']}")
        print("Top 3:", ranked_ids[:3])
        print(f"Query embedding time: {query_seconds:.4f}s")

    print("\n--- Summary ---")
    print(f"Model: {name}")
    print(f"Dimensions: {doc_vectors.shape[1]}")
    print(f"Index embedding time: {indexing_seconds:.4f}s")
    print(f"Mean Recall@3: {np.mean(recalls):.3f}")
    print(f"Mean MRR: {np.mean(reciprocal_ranks):.3f}")

    return {
        "model": name,
        "dimensions": doc_vectors.shape[1],
        "mean_recall_at_3": float(np.mean(recalls)),
        "mean_mrr": float(np.mean(reciprocal_ranks)),
        "index_embedding_seconds": indexing_seconds,
    }


# --------------------------------------------------
# 5. Run the controlled comparison
# --------------------------------------------------

if __name__ == "__main__":
    results = []

    for name, config in MODEL_CONFIGS.items():
        results.append(benchmark_model(name, config))

    print("\n\n===== FINAL COMPARISON =====")

    for result in results:
        print(result)



Loading MiniLM: sentence-transformers/all-MiniLM-L6-v2


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]


Question: How do I troubleshoot an expired bearer token?
Top 3: ['D1', 'D5', 'D2']
Query embedding time: 0.0389s

Question: Why does an authenticated user get forbidden?
Top 3: ['D2', 'D1', 'D5']
Query embedding time: 0.0356s

Question: How can I measure the similarity between text vectors?
Top 3: ['D3', 'D4', 'D5']
Query embedding time: 0.0369s

Question: How does lexical ranking account for rare terms?
Top 3: ['D6', 'D4', 'D5']
Query embedding time: 0.0342s

Question: Why split documents and repeat text near boundaries?
Top 3: ['D5', 'D3', 'D4']
Query embedding time: 0.0360s

--- Summary ---
Model: E5-small
Dimensions: 384
Index embedding time: 0.2139s
Mean Recall@3: 1.000
Mean MRR: 0.900


===== FINAL COMPARISON =====
{'model': 'MiniLM', 'dimensions': 384, 'mean_recall_at_3': 1.0, 'mean_mrr': 0.9, 'index_embedding_seconds': 0.2948770399999603}
{'model': 'E5-small', 'dimensions': 384, 'mean_recall_at_3': 1.0, 'mean_mrr': 0.9, 'index_embedding_seconds': 0.2139022949999685}
